# DOE MASTER FULL PIPELINE V2 — fixed development split, model benchmark, clean outputs

This revision keeps the original-workbook parsing and neutral `WellA`–`WellD` aliases, but simplifies the modeling workflow and output structure.

## Default development design

| Target | Training wells | Fixed validation well | Default status |
|---|---|---|---|
| `hydrate_saturation_vv` | `WellA`, `WellB`, `WellC` | `WellD` | enabled for development |
| `water_saturation_vv` | `WellC` | `WellD` | disabled pending `S_wr`/`Swr` equivalence review |

Every candidate algorithm is evaluated on the **same fixed validation well**. The notebook does not rotate through every held-out well during routine development. A complete leave-one-well-out audit is available behind `RUN_FINAL_CROSS_WELL_AUDIT = True` and should be run only after the model design is frozen.

## Model ladder

- equal-well mean and median baselines
- Ridge
- Elastic Net
- Random Forest
- Extra Trees
- Gradient Boosting

Candidate models remain in memory. Only the selected final model is persisted.

## Default output structure

Each run writes one timestamped folder:

```text
outputs_runtime/runs/<run_id>/
├── model_results.xlsx
├── all_predictions.parquet          # falls back to .csv.gz if Parquet support is absent
├── paper_figures.pdf
├── run_manifest.json
└── north_slope_validation_bundle.zip
```

The review ZIP excludes row-level predictions, source workbook rows, and fitted models. Fitted final models remain local under `models_runtime/runs/<run_id>/`.

## Scientific boundary

- Depth remains excluded from predictors by default.
- Feature selection uses training wells only; validation-well coverage is recorded for audit but never used to decide the feature panel.
- Rows must meet a minimum feature-completeness threshold before scoring.
- NMR porosity remains excluded by default.
- `A090`/`AF90` are not treated as resistivity unless explicitly enabled.
- Tree P10–P90 values are labeled **ensemble spread**, not calibrated predictive intervals.
- Repeated tuning against `WellD` turns it into a development-validation well, not an untouched final test well.


## 0. Imports and one-place run configuration

Edit this cell for normal use. The default routine run is one fixed hydrate split (`WellA` + `WellB` + `WellC` → `WellD`), one model benchmark, and one selected final model.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import platform
import re
import sys
import warnings
import zipfile

import joblib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# =============================================================================
# Source configuration
# =============================================================================

DEFAULT_DATA_DIR = Path.home() / "Downloads" / "Northslopedatasets06052026"
DATA_DIR = Path(os.environ.get("NORTH_SLOPE_DATA_DIR", DEFAULT_DATA_DIR)).expanduser()

SOURCE_SPECS: list[dict[str, Any]] = [
    {
        "well_alias": "WellA",
        "filename": "2L-38_input_Mallik.xlsx",
        "sheet_name": None,
        "source_label": "Mallik 2L-38",
        "default_depth_unit": "m",
        "default_density_unit": "kg/m3",
        "default_caliper_unit": "mm",
    },
    {
        "well_alias": "WellB",
        "filename": "5L-38_input_Mallik.xlsx",
        "sheet_name": None,
        "source_label": "Mallik 5L-38",
        "default_depth_unit": "m",
        "default_density_unit": "kg/m3",
        "default_caliper_unit": "mm",
    },
    {
        "well_alias": "WellC",
        "filename": "MtElbert_Ignik_input_ANS.xlsx",
        "sheet_name": "MTE",
        "source_label": "MTE",
        "default_depth_unit": "ft",
        "default_density_unit": "g/cc",
        "default_caliper_unit": "in",
    },
    {
        "well_alias": "WellD",
        "filename": "MtElbert_Ignik_input_ANS.xlsx",
        "sheet_name": "IGS",
        "source_label": "IGS",
        "default_depth_unit": "ft",
        "default_density_unit": "g/cc",
        "default_caliper_unit": "in",
    },
]

CURATED_GLOB = "curated_dataset*.xlsx"
REFINED_SHEETS = ("MTE_refined", "IGS_refined")

# =============================================================================
# One routine iteration: fixed training and validation wells
# =============================================================================

TARGET_CONFIG: dict[str, dict[str, Any]] = {
    "hydrate_saturation_vv": {
        "enabled": True,
        "train_wells": ("WellA", "WellB", "WellC"),
        "validation_well": "WellD",
        "contract_status": "provisional_development_only__confirm_Sgh_S_h_Sh_equivalence_before_claims",
    },
    "water_saturation_vv": {
        "enabled": False,
        "train_wells": ("WellC",),
        "validation_well": "WellD",
        "contract_status": "blocked_pending_S_wr_Swr_equivalence_review",
    },
}
TARGET_COLUMNS = tuple(TARGET_CONFIG)
ENABLED_TARGETS = tuple(target for target, config in TARGET_CONFIG.items() if config["enabled"])

FEATURE_PANEL = "conservative"  # conservative, strict_independent, or extended
CANDIDATE_MODELS = (
    "mean_baseline",
    "median_baseline",
    "ridge",
    "elastic_net",
    "random_forest",
    "extra_trees",
    "gradient_boosting",
)

RUN_FINAL_CROSS_WELL_AUDIT = False  # turn on once, after model design is frozen
FIT_FINAL_MODELS = True
CREATE_REVIEW_BUNDLE = True
WRITE_STANDARDIZED_TABLE = False
WRITE_PNG_FIGURES = False

RANDOM_STATE = 42
N_JOBS = -1
MIN_TARGET_ROWS = 20
MIN_FEATURE_COVERAGE = 0.20
MIN_COMMON_FEATURES = 3
MIN_FEATURES_PER_ROW = 3
MIN_FEATURE_FRACTION_PER_ROW = 0.50
PERMUTATION_REPEATS = 15
EQUALIZE_WELL_WEIGHTS = True
CLIP_SATURATION_PREDICTIONS = True
USE_DEPTH_AS_FEATURE = False
ALLOW_PROVISIONAL_A090_AS_RESISTIVITY = False
ALLOW_NMR_POROSITY_AS_FEATURE = False

# =============================================================================
# Timestamped outputs: no deleting an existing workspace folder
# =============================================================================

RUN_ID = os.environ.get(
    "NORTH_SLOPE_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")[:-3],
)
OUTPUT_ROOT = Path.cwd() / "outputs_runtime"
MODEL_ROOT = Path.cwd() / "models_runtime"
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID
MODEL_RUN_ROOT = MODEL_ROOT / "runs" / RUN_ID

RESULTS_WORKBOOK = RUN_ROOT / "model_results.xlsx"
PREDICTIONS_PARQUET = RUN_ROOT / "all_predictions.parquet"
FIGURES_PDF = RUN_ROOT / "paper_figures.pdf"
MANIFEST_JSON = RUN_ROOT / "run_manifest.json"
REVIEW_BUNDLE_ZIP = RUN_ROOT / "north_slope_validation_bundle.zip"
LATEST_RUN_JSON = OUTPUT_ROOT / "latest_run.json"


## 1. Canonical fields and leakage policy

The source registry remains compatible with the existing workbook layouts. V2 adds named feature panels so sensitivity testing is explicit rather than silently changing predictors.

In [ ]:
# =============================================================================
# Canonical schema
# =============================================================================

def normalize_token(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return "".join(character for character in str(value).strip().lower() if character.isalnum())


ALIASES: dict[str, tuple[str, ...]] = {
    "depth": (
        "depth", "depth_ft", "depth, ft", "depth ft", "depth m", "depth_m", "dept",
        "true depth", "measured depth", "md", "tvd",
    ),
    "rhob_g_cc": (
        "rho_b", "rhob", "density_gpcc", "density_gcpcc", "density g/cc", "density gpcc",
        "bulk density", "density",
    ),
    "density_porosity_vv": (
        "phi_porosity", "dphi", "phi_den", "density porosity", "density_porosity",
    ),
    "neutron_porosity_vv": ("nphi", "phi_neut", "neutron porosity"),
    "nmr_porosity_vv": ("nmrphi", "phi_nmr", "nmr porosity"),
    "gr_api": ("gr", "gamma ray", "gamma_ray"),
    "caliper_in": ("caliper", "cal1", "cali"),
    "differential_caliper_mm": ("differential caliper", "differential_caliper", "diff caliper"),
    "rt_ohm_m": (
        "res", "rt", "deep formation resistivity", "apparent resistivity", "a090", "af90",
        "computed focusing mode 5", "resistivity",
    ),
    "vp_m_s": ("vp", "velp", "compressional wave velocity", "compressional velocity"),
    "vs_m_s": ("vs", "vs1", "shear wave velocity", "shear velocity"),
    "vp_vs_ratio": ("ratio vp/vs", "ratio vpvs", "vp/vs", "vp_vs_ratio", "ratio of velocities"),
    "acoustic_impedance_source": ("impedance", "acoustic impedance", "rho_b*vp", "rhob*vp"),
    "hydrate_saturation_vv": (
        "sgh", "s_h", "sh", "hydrate saturation", "hydrate_saturation", "hydrate sat",
        "nmr_sat", "gas hydrate saturation",
    ),
    "water_saturation_vv": (
        "s_wr", "swr", "swirr", "irreducible water saturation", "residual water saturation",
    ),
    "alignment_depth_unit_d": ("depths_unitd", "unit d depth", "unitd depth"),
    "alignment_depth_unit_c": ("depths_unitc", "unit c depth", "unitc depth"),
    "aligned_depth": ("depth correspondence at ml data", "aligned depth", "depth correspondence"),
}

ALIAS_LOOKUP: dict[str, str] = {}
for canonical_name, source_aliases in ALIASES.items():
    for alias in (canonical_name, *source_aliases):
        ALIAS_LOOKUP[normalize_token(alias)] = canonical_name

TARGET_ALIAS_TOKENS = {
    token for token, canonical in ALIAS_LOOKUP.items()
    if canonical in TARGET_COLUMNS
}

FEATURE_PANELS: dict[str, tuple[str, ...]] = {
    # Removes duplicate engineered forms while retaining broadly available log families.
    "conservative": (
        "rhob_g_cc",
        "neutron_porosity_vv",
        "gr_api",
        "caliper_in",
        "differential_caliper_mm",
        "log10_rt_ohm_m",
        "vp_m_s",
        "vs_m_s",
    ),
    # Excludes density/NMR-derived quantities for a stricter target-independence sensitivity run.
    "strict_independent": (
        "neutron_porosity_vv",
        "gr_api",
        "caliper_in",
        "differential_caliper_mm",
        "log10_rt_ohm_m",
        "vp_m_s",
        "vs_m_s",
    ),
    # Retains the broader legacy panel for an explicitly labeled sensitivity comparison.
    "extended": (
        "rhob_g_cc",
        "density_porosity_vv",
        "neutron_porosity_vv",
        "gr_api",
        "caliper_in",
        "differential_caliper_mm",
        "rt_ohm_m",
        "vp_m_s",
        "vs_m_s",
        "vp_vs_ratio",
        "acoustic_impedance_kg_m2_s",
        "log10_rt_ohm_m",
    ),
}
if FEATURE_PANEL not in FEATURE_PANELS:
    raise ValueError(f"Unknown FEATURE_PANEL={FEATURE_PANEL!r}. Choose from {tuple(FEATURE_PANELS)}")
MODEL_FEATURE_CANDIDATES = FEATURE_PANELS[FEATURE_PANEL]
if ALLOW_NMR_POROSITY_AS_FEATURE:
    MODEL_FEATURE_CANDIDATES = MODEL_FEATURE_CANDIDATES + (
        "nmr_porosity_vv",
        "nmr_density_separation_vv",
    )

ALIGNMENT_CANONICAL_COLUMNS = {
    "alignment_depth_unit_d", "alignment_depth_unit_c", "aligned_depth"
}

ROLE_TOKENS = {"mlinput", "groundtruth", "outlierremoval", "depth", "qc", "target"}


def canonical_for_header(value: Any) -> str | None:
    token = normalize_token(value)
    canonical = ALIAS_LOOKUP.get(token)
    if token in {"a090", "af90"} and not ALLOW_PROVISIONAL_A090_AS_RESISTIVITY:
        return None
    return canonical


def clean_label(value: Any, fallback: str) -> str:
    try:
        missing = pd.isna(value)
    except Exception:
        missing = False
    text = "" if missing else str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text or fallback


def dedupe_labels(labels: Iterable[str]) -> list[str]:
    seen: dict[str, int] = {}
    result: list[str] = []
    for label in labels:
        count = seen.get(label, 0)
        seen[label] = count + 1
        result.append(label if count == 0 else f"{label}__{count + 1}")
    return result


def numeric_series(values: pd.Series) -> pd.Series:
    cleaned = values.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False)
    cleaned = cleaned.replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan})
    return pd.to_numeric(cleaned, errors="coerce").replace([np.inf, -np.inf], np.nan)


## 2. Workbook parsing and unit-aware standardization

The parser detects the real mnemonic row, locates the first numeric data row, records source role/unit/description metadata, preserves original depth, and converts only documented or strongly inferred units.

In [ ]:
# =============================================================================
# Workbook parsing
# =============================================================================

def header_row_score(row: pd.Series) -> tuple[int, int, int]:
    tokens = [normalize_token(value) for value in row.tolist() if normalize_token(value)]
    exact = sum(token in ALIAS_LOOKUP for token in tokens)
    target = sum(token in TARGET_ALIAS_TOKENS for token in tokens)
    keyword = sum(
        any(part in token for part in (
            "depth", "density", "porosity", "caliper", "resist", "gamma", "velocity",
            "impedance", "saturation",
        ))
        for token in tokens
    )
    return exact * 10 + target * 3 + keyword, exact, len(tokens)


def detect_header_layout(raw: pd.DataFrame, max_scan_rows: int = 10) -> dict[str, Any]:
    if raw.empty:
        raise ValueError("Sheet is empty.")
    scan_count = min(max_scan_rows, len(raw))
    scores = [header_row_score(raw.iloc[row_index]) for row_index in range(scan_count)]
    header_index = max(range(scan_count), key=lambda index: scores[index])
    if scores[header_index][1] < 3:
        raise ValueError(f"Could not identify a reliable mnemonic header row. Scores: {scores}")

    header_values = [
        clean_label(value, f"unnamed_{column_index}")
        for column_index, value in enumerate(raw.iloc[header_index].tolist())
    ]
    meaningful_columns = [index for index, label in enumerate(header_values) if not label.startswith("unnamed_")]

    data_start = None
    for row_index in range(header_index + 1, min(len(raw), header_index + 12)):
        row = raw.iloc[row_index, meaningful_columns] if meaningful_columns else raw.iloc[row_index]
        nonempty = int(row.notna().sum())
        numeric = int(numeric_series(row).notna().sum())
        if numeric >= 3 and numeric / max(nonempty, 1) >= 0.45:
            data_start = row_index
            break
    if data_start is None:
        raise ValueError("Could not identify the first numeric data row after the header.")

    role_index = None
    if header_index > 0:
        role_tokens = {normalize_token(value) for value in raw.iloc[header_index - 1].tolist()}
        if role_tokens & ROLE_TOKENS:
            role_index = header_index - 1

    unit_index = None
    description_index = None
    for row_index in range(header_index + 1, data_start):
        tokens = [normalize_token(value) for value in raw.iloc[row_index].tolist() if normalize_token(value)]
        unit_hits = sum(
            token in {"m", "ft", "kgm3", "gcc", "gcm3", "api", "mm", "in", "ohmm", "ms", "kms", "ratio", "nmrsat"}
            or any(unit in token for unit in ("kgm3", "ohmm", "ms", "kms", "api", "mm", "inch", "feet"))
            for token in tokens
        )
        if unit_hits >= 2 and unit_index is None:
            unit_index = row_index
        description_index = row_index

    return {
        "header_index": header_index,
        "data_start": data_start,
        "role_index": role_index,
        "unit_index": unit_index,
        "description_index": description_index,
        "header_score": scores[header_index][0],
        "recognized_headers": scores[header_index][1],
    }


def choose_best_sheet(path: Path) -> tuple[str, pd.DataFrame, dict[str, Any]]:
    with pd.ExcelFile(path) as excel:
        sheet_names = list(excel.sheet_names)
    candidates: list[tuple[int, int, str, pd.DataFrame, dict[str, Any]]] = []
    errors: list[str] = []
    for sheet_name in sheet_names:
        if "refined" in normalize_token(sheet_name):
            continue
        try:
            raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
            layout = detect_header_layout(raw)
            candidates.append((layout["recognized_headers"], len(raw), sheet_name, raw, layout))
        except Exception as exc:
            errors.append(f"{sheet_name}: {exc}")
    if not candidates:
        raise ValueError(f"No model-input sheet could be identified in {path.name}. Details: {errors}")
    _, _, sheet_name, raw, layout = sorted(candidates, key=lambda item: (item[0], item[1]), reverse=True)[0]
    return sheet_name, raw, layout


def read_source_sheet(path: Path, requested_sheet: str | None) -> tuple[str, pd.DataFrame, dict[str, Any]]:
    if requested_sheet is None:
        return choose_best_sheet(path)
    with pd.ExcelFile(path) as excel:
        matching = {normalize_token(name): name for name in excel.sheet_names}
    actual = matching.get(normalize_token(requested_sheet))
    if actual is None:
        raise ValueError(f"Sheet {requested_sheet!r} is missing from {path.name}.")
    raw = pd.read_excel(path, sheet_name=actual, header=None)
    return actual, raw, detect_header_layout(raw)


def infer_unit(header: str, unit: str, default: str, canonical: str) -> str:
    combined = f"{header} {unit}".lower().replace("³", "3")
    token = normalize_token(combined)
    if canonical == "depth":
        if "ft" in combined or "feet" in combined:
            return "ft"
        if re.search(r"(^|\s)m($|\s)", combined) and "mm" not in combined and "/s" not in combined:
            return "m"
    if canonical == "rhob_g_cc":
        if "kg/m3" in combined or "kgm3" in token:
            return "kg/m3"
        if any(value in combined for value in ("g/cc", "g/cm3", "gpcc", "gcpcc")):
            return "g/cc"
    if canonical in {"caliper_in", "differential_caliper_mm"}:
        if "mm" in combined:
            return "mm"
        if any(value in combined for value in ("inch", " in", "in.")):
            return "in"
    if canonical in {"vp_m_s", "vs_m_s"}:
        if "km/s" in combined or "kms" in token:
            return "km/s"
        if "m/s" in combined or token.endswith("ms"):
            return "m/s"
    return default


def convert_fraction(
    values: pd.Series,
    field: str,
    audit: list[dict[str, Any]],
    well_alias: str,
    source_header: str,
) -> pd.Series:
    numeric = numeric_series(values)
    finite = numeric.dropna()
    conversion = "none"
    if not finite.empty:
        q99 = float(finite.quantile(0.99))
        if q99 > 1.5 and q99 <= 100.5:
            numeric = numeric / 100.0
            conversion = "percent_to_fraction"
    audit.append({
        "well_alias": well_alias,
        "field": field,
        "source_header": source_header,
        "source_unit": "unknown_or_fraction",
        "canonical_unit": "fraction",
        "conversion": conversion,
        "source_min": float(finite.min()) if not finite.empty else np.nan,
        "source_max": float(finite.max()) if not finite.empty else np.nan,
    })
    return numeric


def standardize_well(spec: dict[str, Any], data_dir: Path | None = None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    data_dir = Path(data_dir or DATA_DIR)
    well_alias = spec["well_alias"]
    path = data_dir / spec["filename"]
    if not path.exists():
        raise FileNotFoundError(path)
    sheet_name, raw, layout = read_source_sheet(path, spec.get("sheet_name"))

    headers = dedupe_labels([
        clean_label(value, f"unnamed_{column_index}")
        for column_index, value in enumerate(raw.iloc[layout["header_index"]].tolist())
    ])
    roles = (
        [clean_label(value, "") for value in raw.iloc[layout["role_index"]].tolist()]
        if layout["role_index"] is not None else [""] * len(headers)
    )
    units = (
        [clean_label(value, "") for value in raw.iloc[layout["unit_index"]].tolist()]
        if layout["unit_index"] is not None else [""] * len(headers)
    )
    descriptions = (
        [clean_label(value, "") for value in raw.iloc[layout["description_index"]].tolist()]
        if layout["description_index"] is not None else [""] * len(headers)
    )

    data = raw.iloc[layout["data_start"]:].copy().reset_index(drop=True)
    data.columns = headers
    data = data.dropna(axis=0, how="all").dropna(axis=1, how="all")

    mapping_rows: list[dict[str, Any]] = []
    unit_audit: list[dict[str, Any]] = []
    canonical_sources: dict[str, str] = {}
    canonical_series: dict[str, pd.Series] = {}

    for column_index, header in enumerate(headers):
        if header not in data.columns:
            continue
        canonical = canonical_for_header(header)
        role = roles[column_index] if column_index < len(roles) else ""
        unit = units[column_index] if column_index < len(units) else ""
        description = descriptions[column_index] if column_index < len(descriptions) else ""
        mapping_rows.append({
            "well_alias": well_alias,
            "source_workbook": spec["filename"],
            "source_sheet": sheet_name,
            "column_position": column_index + 1,
            "source_header": header,
            "source_role": role,
            "source_unit": unit,
            "source_description": description,
            "canonical_field": canonical or "unmapped",
            "model_permission": (
                "target_only" if canonical in TARGET_COLUMNS else
                "alignment_only" if canonical in ALIGNMENT_CANONICAL_COLUMNS else
                "candidate_feature" if canonical in MODEL_FEATURE_CANDIDATES or canonical in {
                    "rhob_g_cc", "density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv",
                    "gr_api", "caliper_in", "differential_caliper_mm", "rt_ohm_m", "vp_m_s",
                    "vs_m_s", "vp_vs_ratio", "acoustic_impedance_source",
                } else "excluded_or_unmapped"
            ),
        })
        if canonical is None or canonical in ALIGNMENT_CANONICAL_COLUMNS:
            continue
        candidate = numeric_series(data[header])
        if canonical in canonical_series:
            if candidate.notna().sum() > canonical_series[canonical].notna().sum():
                canonical_series[canonical] = candidate
                canonical_sources[canonical] = header
        else:
            canonical_series[canonical] = candidate
            canonical_sources[canonical] = header

    standardized = pd.DataFrame(index=data.index)
    standardized["well_alias"] = well_alias
    standardized["source_label"] = spec["source_label"]
    standardized["source_workbook"] = spec["filename"]
    standardized["source_sheet"] = sheet_name
    standardized["source_row"] = np.arange(layout["data_start"] + 1, layout["data_start"] + 1 + len(data))

    if "depth" not in canonical_series:
        raise ValueError(f"No depth column was mapped for {well_alias} from {path.name}/{sheet_name}.")

    depth_source_header = canonical_sources["depth"]
    depth_column_position = headers.index(depth_source_header)
    depth_unit_text = units[depth_column_position] if depth_column_position < len(units) else ""
    depth_unit = infer_unit(depth_source_header, depth_unit_text, spec["default_depth_unit"], "depth")
    depth_original = numeric_series(data[depth_source_header])
    standardized["depth_original"] = depth_original
    standardized["depth_original_unit"] = depth_unit
    standardized["depth_m"] = depth_original * 0.3048 if depth_unit == "ft" else depth_original
    unit_audit.append({
        "well_alias": well_alias,
        "field": "depth_m",
        "source_header": depth_source_header,
        "source_unit": depth_unit,
        "canonical_unit": "m",
        "conversion": "multiply_0.3048" if depth_unit == "ft" else "identity",
        "source_min": float(depth_original.min()) if depth_original.notna().any() else np.nan,
        "source_max": float(depth_original.max()) if depth_original.notna().any() else np.nan,
    })

    standard_fields = (
        "rhob_g_cc", "density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv",
        "gr_api", "caliper_in", "differential_caliper_mm", "rt_ohm_m", "vp_m_s", "vs_m_s",
        "vp_vs_ratio", "acoustic_impedance_source", "hydrate_saturation_vv", "water_saturation_vv",
    )
    for field in standard_fields:
        if field not in canonical_series:
            continue
        source_header = canonical_sources[field]
        source_position = headers.index(source_header)
        source_unit_text = units[source_position] if source_position < len(units) else ""
        values = canonical_series[field].copy()

        if field == "rhob_g_cc":
            unit = infer_unit(source_header, source_unit_text, spec["default_density_unit"], field)
            finite = values.dropna()
            inferred_by_magnitude = False
            if unit == "g/cc" and not finite.empty and float(finite.median()) > 20:
                unit = "kg/m3"
                inferred_by_magnitude = True
            standardized[field] = values / 1000.0 if unit == "kg/m3" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "g/cc",
                "conversion": "divide_1000" if unit == "kg/m3" else "identity",
                "inferred_by_magnitude": inferred_by_magnitude,
            })
        elif field in {"density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv", *TARGET_COLUMNS}:
            if field in TARGET_COLUMNS:
                standardized[f"{field}_original"] = values
                standardized[f"{field}_source_header"] = source_header
            standardized[field] = convert_fraction(values, field, unit_audit, well_alias, source_header)
        elif field == "caliper_in":
            unit = infer_unit(source_header, source_unit_text, spec["default_caliper_unit"], field)
            finite = values.dropna()
            if unit == "in" and not finite.empty and float(finite.median()) > 50:
                unit = "mm"
            standardized[field] = values / 25.4 if unit == "mm" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "in",
                "conversion": "divide_25.4" if unit == "mm" else "identity",
            })
        elif field == "differential_caliper_mm":
            unit = infer_unit(source_header, source_unit_text, "mm", field)
            standardized[field] = values * 25.4 if unit == "in" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "mm",
                "conversion": "multiply_25.4" if unit == "in" else "identity",
            })
        elif field in {"vp_m_s", "vs_m_s"}:
            unit = infer_unit(source_header, source_unit_text, "m/s", field)
            finite = values.dropna()
            if unit == "m/s" and not finite.empty and 0 < float(finite.median()) < 20:
                unit = "km/s"
            standardized[field] = values * 1000.0 if unit == "km/s" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "m/s",
                "conversion": "multiply_1000" if unit == "km/s" else "identity",
            })
        else:
            standardized[field] = values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": source_unit_text or "unknown", "canonical_unit": "preserved",
                "conversion": "identity",
            })

    if {"vp_m_s", "vs_m_s"}.issubset(standardized.columns):
        ratio = standardized["vp_m_s"] / standardized["vs_m_s"].replace(0, np.nan)
        if "vp_vs_ratio" not in standardized or standardized["vp_vs_ratio"].notna().mean() < 0.20:
            standardized["vp_vs_ratio"] = ratio
        else:
            standardized["vp_vs_ratio_recomputed"] = ratio
    if {"rhob_g_cc", "vp_m_s"}.issubset(standardized.columns):
        standardized["acoustic_impedance_kg_m2_s"] = standardized["rhob_g_cc"] * 1000.0 * standardized["vp_m_s"]
    if "rt_ohm_m" in standardized:
        positive_rt = standardized["rt_ohm_m"].where(standardized["rt_ohm_m"] > 0)
        standardized["log10_rt_ohm_m"] = np.log10(positive_rt)
    if {"density_porosity_vv", "nmr_porosity_vv"}.issubset(standardized.columns):
        standardized["nmr_density_separation_vv"] = standardized["density_porosity_vv"] - standardized["nmr_porosity_vv"]

    numeric_columns = [column for column in standardized.columns if column not in {
        "well_alias", "source_label", "source_workbook", "source_sheet", "depth_original_unit",
        "hydrate_saturation_vv_source_header", "water_saturation_vv_source_header",
    }]
    for column in numeric_columns:
        standardized[column] = pd.to_numeric(standardized[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

    signal_columns = [column for column in standardized.columns if column in MODEL_FEATURE_CANDIDATES or column in TARGET_COLUMNS]
    keep_mask = standardized["depth_m"].notna()
    if signal_columns:
        keep_mask &= standardized[signal_columns].notna().any(axis=1)
    standardized = standardized.loc[keep_mask].copy()
    standardized = standardized.sort_values(["depth_m", "source_row"], kind="stable").reset_index(drop=True)

    layout_record = {
        "well_alias": well_alias,
        "source_workbook": spec["filename"],
        "source_sheet": sheet_name,
        "header_row_excel": layout["header_index"] + 1,
        "data_start_row_excel": layout["data_start"] + 1,
        "role_row_excel": layout["role_index"] + 1 if layout["role_index"] is not None else None,
        "unit_row_excel": layout["unit_index"] + 1 if layout["unit_index"] is not None else None,
        "description_row_excel": layout["description_index"] + 1 if layout["description_index"] is not None else None,
        "recognized_header_count": layout["recognized_headers"],
        "standardized_rows": len(standardized),
    }
    return standardized, pd.DataFrame(mapping_rows), pd.DataFrame(unit_audit), layout_record


def refined_sheet_inventory(path: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if not path.exists():
        return pd.DataFrame(rows)
    with pd.ExcelFile(path) as excel:
        names = {normalize_token(name): name for name in excel.sheet_names}
    for requested in REFINED_SHEETS:
        actual = names.get(normalize_token(requested))
        if actual is None:
            rows.append({"source_workbook": path.name, "source_sheet": requested, "status": "missing"})
            continue
        raw = pd.read_excel(path, sheet_name=actual, header=None)
        target_like_cells = sum(
            normalize_token(value) in TARGET_ALIAS_TOKENS
            for value in raw.head(12).to_numpy().ravel().tolist()
        )
        rows.append({
            "source_workbook": path.name,
            "source_sheet": actual,
            "status": "excluded_alignment_qc_only",
            "rows": len(raw),
            "columns": len(raw.columns),
            "target_like_header_cells_first_12_rows": target_like_cells,
            "reason": "Refined/alignment sheets are not counted as wells and are never used as training tables.",
        })
    return pd.DataFrame(rows)

## 3. Standardized WellA–WellD tables and hard-stop audits

V2 keeps all standardization and audit tables in memory and later writes them as sheets in one Excel workbook. It no longer creates a directory of small CSV files during preflight.

In [ ]:
# =============================================================================
# Standardization and audits — in memory until the final consolidated export
# =============================================================================

def prepare_run_directories() -> None:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    MODEL_RUN_ROOT.mkdir(parents=True, exist_ok=True)


def standardize_all_sources(data_dir: Path | None = None) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    data_dir = Path(data_dir or DATA_DIR)
    prepare_run_directories()
    missing = [spec["filename"] for spec in SOURCE_SPECS if not (data_dir / spec["filename"]).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing original workbooks in " + str(data_dir) + ": " + ", ".join(missing)
        )

    ignored_files = sorted(path.name for path in data_dir.glob(CURATED_GLOB))
    ignored_df = pd.DataFrame([
        {"filename": name, "status": "ignored", "reason": "Curated/normalized duplicate; original workbook is authoritative."}
        for name in ignored_files
    ])

    well_frames: dict[str, pd.DataFrame] = {}
    mappings: list[pd.DataFrame] = []
    units: list[pd.DataFrame] = []
    layouts: list[dict[str, Any]] = []
    for spec in SOURCE_SPECS:
        frame, mapping, unit_audit, layout = standardize_well(spec, data_dir=data_dir)
        well_alias = spec["well_alias"]
        well_frames[well_alias] = frame
        mappings.append(mapping)
        units.append(unit_audit)
        layouts.append(layout)

    combined = pd.concat(well_frames.values(), ignore_index=True, sort=False)
    mapping_df = pd.concat(mappings, ignore_index=True, sort=False)
    unit_df = pd.concat(units, ignore_index=True, sort=False)
    layout_df = pd.DataFrame(layouts)
    refined = refined_sheet_inventory(data_dir / "MtElbert_Ignik_input_ANS.xlsx")

    return well_frames, {
        "combined": combined,
        "column_mapping": mapping_df,
        "unit_conversion": unit_df,
        "source_layout": layout_df,
        "refined_sheet_inventory": refined,
        "ignored_inputs": ignored_df,
    }


def build_target_and_readiness_audits(
    well_frames: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    target_rows: list[dict[str, Any]] = []
    readiness_rows: list[dict[str, Any]] = []
    feature_rows: list[dict[str, Any]] = []
    blocking_messages: list[str] = []

    for well_alias, frame in well_frames.items():
        depth = frame["depth_m"].dropna()
        suspicious_depth = depth.empty or float(depth.max()) <= 10 or float(depth.max() - depth.min()) <= 1
        if suspicious_depth:
            blocking_messages.append(f"{well_alias}: depth appears normalized or invalid.")
        readiness_rows.append({
            "well_alias": well_alias,
            "rows": len(frame),
            "depth_non_null": int(frame["depth_m"].notna().sum()),
            "depth_min_m": float(depth.min()) if not depth.empty else np.nan,
            "depth_max_m": float(depth.max()) if not depth.empty else np.nan,
            "depth_span_m": float(depth.max() - depth.min()) if not depth.empty else np.nan,
            "depth_monotonic": bool(depth.is_monotonic_increasing),
            "depth_duplicate_rows": int(frame["depth_m"].duplicated().sum()),
            "suspicious_normalized_depth": suspicious_depth,
        })
        for target in TARGET_COLUMNS:
            series = frame[target] if target in frame else pd.Series(index=frame.index, dtype=float)
            valid = series.dropna()
            out_of_range = int(((valid < -0.001) | (valid > 1.001)).sum()) if not valid.empty else 0
            constant = bool(valid.nunique() <= 1) if not valid.empty else False
            eligible = bool(valid.size >= MIN_TARGET_ROWS and not constant and out_of_range == 0)
            source_header_column = f"{target}_source_header"
            source_headers = (
                ", ".join(sorted(frame[source_header_column].dropna().astype(str).unique()))
                if source_header_column in frame else ""
            )
            target_rows.append({
                "well_alias": well_alias,
                "target": target,
                "model_enabled": bool(TARGET_CONFIG[target]["enabled"]),
                "target_contract_status": TARGET_CONFIG[target]["contract_status"],
                "source_headers": source_headers,
                "rows": len(frame),
                "target_rows": int(valid.size),
                "target_fraction": float(valid.size / max(len(frame), 1)),
                "target_min": float(valid.min()) if not valid.empty else np.nan,
                "target_max": float(valid.max()) if not valid.empty else np.nan,
                "target_mean": float(valid.mean()) if not valid.empty else np.nan,
                "unique_values": int(valid.nunique()),
                "constant_target": constant,
                "out_of_range_rows": out_of_range,
                "eligible_for_training": eligible,
            })
            if valid.size >= MIN_TARGET_ROWS and out_of_range > 0:
                blocking_messages.append(f"{well_alias}/{target}: target values remain outside [0, 1].")
        for feature in MODEL_FEATURE_CANDIDATES:
            coverage = float(frame[feature].notna().mean()) if feature in frame else 0.0
            feature_rows.append({
                "well_alias": well_alias,
                "feature_panel": FEATURE_PANEL,
                "feature": feature,
                "coverage": coverage,
                "available": bool(coverage >= MIN_FEATURE_COVERAGE),
                "minimum": float(frame[feature].min()) if feature in frame and frame[feature].notna().any() else np.nan,
                "maximum": float(frame[feature].max()) if feature in frame and frame[feature].notna().any() else np.nan,
            })

    target_audit = pd.DataFrame(target_rows)
    readiness = pd.DataFrame(readiness_rows)
    feature_coverage = pd.DataFrame(feature_rows)

    leakage_rows: list[dict[str, Any]] = []
    audit_features = sorted(set().union(*FEATURE_PANELS.values(), {"nmr_porosity_vv", "nmr_density_separation_vv"}))
    for target in TARGET_COLUMNS:
        for feature in audit_features:
            if feature in {"nmr_porosity_vv", "nmr_density_separation_vv"} and not ALLOW_NMR_POROSITY_AS_FEATURE:
                decision = "excluded"
                reason = "NMR-derived predictor excluded by default because of circular target-reconstruction risk."
            elif target == "hydrate_saturation_vv" and feature in {"rhob_g_cc", "density_porosity_vv"}:
                decision = "allowed_with_derivation_review"
                reason = "Density may overlap the stated NMR-density target derivation; retain only for labeled sensitivity analysis."
            elif feature in MODEL_FEATURE_CANDIDATES:
                decision = "allowed"
                reason = f"Included in the named {FEATURE_PANEL!r} feature panel."
            else:
                decision = "excluded_from_active_panel"
                reason = f"Not part of the named {FEATURE_PANEL!r} feature panel."
            leakage_rows.append({
                "target": target,
                "candidate_feature": feature,
                "decision": decision,
                "reason": reason,
            })
        leakage_rows.extend([
            {"target": target, "candidate_feature": other_target, "decision": "excluded", "reason": "target_or_target_like_leakage"}
            for other_target in TARGET_COLUMNS
        ])
        leakage_rows.extend([
            {"target": target, "candidate_feature": field, "decision": "excluded", "reason": "identifier_alignment_or_context"}
            for field in ("well_alias", "source_workbook", "source_sheet", "source_row", "depth_original", "depth_m")
        ])
    leakage = pd.DataFrame(leakage_rows)

    if blocking_messages:
        raise RuntimeError("Standardization audit blocked modeling:\n- " + "\n- ".join(blocking_messages))
    return target_audit, readiness, feature_coverage, leakage


## 4. Fixed-split model benchmark and optional final audit

Feature selection is based only on the configured training wells. All algorithms see the same training rows, validation well, feature panel, completeness rule, and equal-well weighting. Candidate models are not written to disk.

In [ ]:
# =============================================================================
# Modeling helpers
# =============================================================================

def _target_is_trainable(frame: pd.DataFrame, target: str) -> bool:
    if target not in frame:
        return False
    valid = frame[target].dropna()
    return bool(len(valid) >= MIN_TARGET_ROWS and valid.nunique() > 1)


def select_training_features(
    well_frames: dict[str, pd.DataFrame],
    training_wells: list[str],
    validation_well: str | None = None,
) -> tuple[list[str], pd.DataFrame]:
    """Select features from training coverage only; validation coverage is audit-only."""
    rows: list[dict[str, Any]] = []
    selected: list[str] = []
    for feature in MODEL_FEATURE_CANDIDATES:
        training_coverage = {
            well: float(well_frames[well][feature].notna().mean()) if feature in well_frames[well] else 0.0
            for well in training_wells
        }
        training_parts = [
            well_frames[well][feature] for well in training_wells if feature in well_frames[well]
        ]
        training_values = pd.concat(training_parts, ignore_index=True) if training_parts else pd.Series(dtype=float)
        training_non_null = training_values.dropna()
        reason = ""
        if any(value < MIN_FEATURE_COVERAGE for value in training_coverage.values()):
            reason = "insufficient_training_well_coverage"
        elif training_non_null.nunique() <= 1:
            reason = "constant_or_empty_in_training_wells"
        else:
            selected.append(feature)
        row: dict[str, Any] = {
            "feature_panel": FEATURE_PANEL,
            "feature": feature,
            "decision": "included" if not reason else "excluded",
            "reason": reason,
            "selection_used_validation_well": False,
            "training_non_null_rows": int(training_non_null.size),
            "training_unique_values": int(training_non_null.nunique()),
        }
        row.update({f"training_coverage_{well}": value for well, value in training_coverage.items()})
        if validation_well is not None:
            row["validation_well"] = validation_well
            row["validation_coverage_audit_only"] = (
                float(well_frames[validation_well][feature].notna().mean())
                if feature in well_frames[validation_well] else 0.0
            )
        rows.append(row)
    if USE_DEPTH_AS_FEATURE:
        selected.append("depth_m")
    if len(selected) < MIN_COMMON_FEATURES:
        raise RuntimeError(
            f"Only {len(selected)} training-selected features were available for {training_wells}. "
            "Review source mappings or choose a compatible named feature panel."
        )
    return selected, pd.DataFrame(rows)


def row_score_mask(frame: pd.DataFrame, feature_columns: list[str]) -> tuple[pd.Series, pd.Series, int]:
    feature_frame = frame.reindex(columns=feature_columns)
    present_count = feature_frame.notna().sum(axis=1)
    required_count = min(
        len(feature_columns),
        max(MIN_FEATURES_PER_ROW, int(math.ceil(len(feature_columns) * MIN_FEATURE_FRACTION_PER_ROW))),
    )
    return present_count >= required_count, present_count, required_count


def build_training_data(
    well_frames: dict[str, pd.DataFrame],
    training_wells: list[str],
    target: str,
    feature_columns: list[str],
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for well in training_wells:
        frame = well_frames[well]
        scoreable, present_count, required_count = row_score_mask(frame, feature_columns)
        mask = frame[target].notna() & scoreable
        part = frame.loc[mask, feature_columns + [target]].copy()
        part["well_alias"] = well
        part["feature_count_present"] = present_count.loc[mask].to_numpy()
        part["feature_count_required"] = required_count
        parts.append(part)
    if not parts:
        raise RuntimeError(f"No training rows were available for {target}.")
    train_data = pd.concat(parts, ignore_index=True, sort=False)
    if len(train_data) < MIN_TARGET_ROWS:
        raise RuntimeError(f"Only {len(train_data)} scoreable training rows were available for {target}.")
    return train_data


def equal_well_sample_weights(train_data: pd.DataFrame) -> np.ndarray:
    if not EQUALIZE_WELL_WEIGHTS:
        return np.ones(len(train_data), dtype=float)
    counts = train_data["well_alias"].value_counts()
    weights = train_data["well_alias"].map(lambda well: 1.0 / counts[well]).to_numpy(dtype=float)
    return weights / np.mean(weights)


def make_model(algorithm: str) -> Pipeline:
    steps: list[tuple[str, Any]] = [("imputer", SimpleImputer(strategy="median"))]
    if algorithm in {"ridge", "elastic_net"}:
        steps.append(("scaler", StandardScaler()))

    if algorithm == "mean_baseline":
        estimator = DummyRegressor(strategy="mean")
    elif algorithm == "median_baseline":
        estimator = DummyRegressor(strategy="median")
    elif algorithm == "ridge":
        estimator = Ridge(alpha=2.0)
    elif algorithm == "elastic_net":
        estimator = ElasticNet(alpha=0.001, l1_ratio=0.25, max_iter=20000, random_state=RANDOM_STATE)
    elif algorithm == "random_forest":
        estimator = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=5,
            max_features=0.75,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
    elif algorithm == "extra_trees":
        estimator = ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            max_features=0.75,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
    elif algorithm == "gradient_boosting":
        estimator = GradientBoostingRegressor(
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            loss="huber",
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")
    steps.append(("model", estimator))
    return Pipeline(steps)


def fit_model(
    model: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    sample_weight: np.ndarray,
) -> bool:
    estimator = model.named_steps["model"]
    try:
        supports_sample_weight = "sample_weight" in inspect.signature(estimator.fit).parameters
    except (TypeError, ValueError):
        supports_sample_weight = False
    fit_parameters = {"model__sample_weight": sample_weight} if supports_sample_weight else {}
    model.fit(X, y, **fit_parameters)
    return supports_sample_weight


def bounded_predictions(raw_predictions: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    raw = np.asarray(raw_predictions, dtype=float)
    bounded = np.clip(raw, 0.0, 1.0) if CLIP_SATURATION_PREDICTIONS else raw.copy()
    clipped = ~np.isclose(raw, bounded, equal_nan=True)
    return bounded, clipped


def ensemble_spread(model: Pipeline, X: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Tree-to-tree spread only; this is not a calibrated predictive interval."""
    estimator = model.named_steps["model"]
    if not hasattr(estimator, "estimators_") or isinstance(estimator.estimators_, np.ndarray):
        missing = np.full(len(X), np.nan)
        return missing, missing
    transformed: Any = X
    for name, step in model.named_steps.items():
        if name == "model":
            break
        transformed = step.transform(transformed)
    tree_predictions = np.vstack([tree.predict(transformed) for tree in estimator.estimators_])
    return np.quantile(tree_predictions, 0.10, axis=0), np.quantile(tree_predictions, 0.90, axis=0)


def metric_record(y_true: pd.Series | np.ndarray, y_pred_raw: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:
    true = pd.to_numeric(pd.Series(y_true), errors="coerce").to_numpy(dtype=float)
    raw = np.asarray(y_pred_raw, dtype=float)
    bounded = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(true) & np.isfinite(raw) & np.isfinite(bounded)
    true = true[valid]
    raw = raw[valid]
    bounded = bounded[valid]
    if len(true) == 0:
        return {
            "scored_rows": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "bias": np.nan,
            "raw_mae": np.nan,
            "raw_rmse": np.nan,
            "raw_r2": np.nan,
            "raw_bias": np.nan,
            "prediction_clip_fraction": np.nan,
        }

    def _r2(values: np.ndarray) -> float:
        return float(r2_score(true, values)) if len(true) >= 2 and np.nanstd(true) > 0 else np.nan

    return {
        "scored_rows": int(len(true)),
        "mae": float(mean_absolute_error(true, bounded)),
        "rmse": float(np.sqrt(mean_squared_error(true, bounded))),
        "r2": _r2(bounded),
        "bias": float(np.mean(bounded - true)),
        "raw_mae": float(mean_absolute_error(true, raw)),
        "raw_rmse": float(np.sqrt(mean_squared_error(true, raw))),
        "raw_r2": _r2(raw),
        "raw_bias": float(np.mean(raw - true)),
        "prediction_clip_fraction": float(np.mean(~np.isclose(raw, bounded))),
    }


def feature_shift_records(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    feature_columns: list[str],
    target: str,
    model_run: str,
    test_well: str,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for feature in feature_columns:
        train_values = train_frame[feature].dropna()
        test_values = test_frame[feature].dropna()
        if train_values.empty or test_values.empty:
            train_min = train_max = outside_fraction = normalized_median_shift = np.nan
        else:
            train_min = float(train_values.min())
            train_max = float(train_values.max())
            outside_fraction = float(((test_values < train_min) | (test_values > train_max)).mean())
            train_iqr = float(train_values.quantile(0.75) - train_values.quantile(0.25))
            normalized_median_shift = (
                float(abs(test_values.median() - train_values.median()) / train_iqr)
                if train_iqr > 0 else np.nan
            )
        rows.append({
            "model_run": model_run,
            "target": target,
            "test_well": test_well,
            "feature": feature,
            "train_min": train_min,
            "train_max": train_max,
            "test_min": float(test_values.min()) if not test_values.empty else np.nan,
            "test_max": float(test_values.max()) if not test_values.empty else np.nan,
            "test_fraction_outside_training_range": outside_fraction,
            "absolute_median_shift_in_training_iqr": normalized_median_shift,
        })
    return rows


def heldout_permutation_importance(
    model: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    feature_columns: list[str],
    target: str,
    validation_well: str,
    algorithm: str,
) -> pd.DataFrame:
    valid = y.notna()
    if valid.sum() < 2 or algorithm in {"mean_baseline", "median_baseline"}:
        return pd.DataFrame()
    result = permutation_importance(
        model,
        X.loc[valid, feature_columns],
        y.loc[valid],
        scoring="neg_root_mean_squared_error",
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    return pd.DataFrame({
        "model_run": "fixed_development_split",
        "target": target,
        "validation_well": validation_well,
        "algorithm": algorithm,
        "feature": feature_columns,
        "importance_mean_rmse_increase": result.importances_mean,
        "importance_std": result.importances_std,
        "repeats": PERMUTATION_REPEATS,
    }).sort_values("importance_mean_rmse_increase", ascending=False)


def run_fixed_development_benchmark(well_frames: dict[str, pd.DataFrame]) -> dict[str, Any]:
    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, Any]] = []
    ranking_frames: list[pd.DataFrame] = []
    importance_frames: list[pd.DataFrame] = []
    feature_policy_frames: list[pd.DataFrame] = []
    shift_rows: list[dict[str, Any]] = []
    inventory_rows: list[dict[str, Any]] = []
    selected_specs: dict[str, dict[str, Any]] = {}

    for target in TARGET_COLUMNS:
        config = TARGET_CONFIG[target]
        if not config["enabled"]:
            inventory_rows.append({
                "model_run": "fixed_development_split",
                "target": target,
                "status": "disabled",
                "reason": config["contract_status"],
            })
            continue

        training_wells = list(config["train_wells"])
        validation_well = str(config["validation_well"])
        missing_wells = [well for well in training_wells + [validation_well] if well not in well_frames]
        if missing_wells:
            raise RuntimeError(f"Configured wells are missing for {target}: {missing_wells}")
        untrainable = [well for well in training_wells if not _target_is_trainable(well_frames[well], target)]
        if untrainable:
            raise RuntimeError(f"Configured training wells lack usable {target}: {untrainable}")
        if not _target_is_trainable(well_frames[validation_well], target):
            raise RuntimeError(f"Configured validation well {validation_well} lacks usable {target}.")

        feature_columns, feature_policy = select_training_features(
            well_frames, training_wells, validation_well=validation_well
        )
        feature_policy.insert(0, "train_wells", ",".join(training_wells))
        feature_policy.insert(0, "target", target)
        feature_policy.insert(0, "model_run", "fixed_development_split")
        feature_policy_frames.append(feature_policy)

        train_data = build_training_data(well_frames, training_wells, target, feature_columns)
        X_train = train_data[feature_columns]
        y_train = train_data[target]
        sample_weight = equal_well_sample_weights(train_data)

        validation_frame = well_frames[validation_well]
        scoreable, present_count, required_count = row_score_mask(validation_frame, feature_columns)
        X_validation = validation_frame.loc[scoreable, feature_columns].copy()
        if X_validation.empty:
            raise RuntimeError(f"Validation well {validation_well} has no scoreable rows for {target}.")
        y_validation = validation_frame.loc[scoreable, target].copy()

        candidate_models: dict[str, Pipeline] = {}
        target_metric_rows: list[dict[str, Any]] = []
        target_prediction_frames: list[pd.DataFrame] = []

        for algorithm in CANDIDATE_MODELS:
            model = make_model(algorithm)
            supports_weight = fit_model(model, X_train, y_train, sample_weight)
            raw_predictions = model.predict(X_validation)
            predictions, clipped_flags = bounded_predictions(raw_predictions)
            p10, p90 = ensemble_spread(model, X_validation)
            if CLIP_SATURATION_PREDICTIONS:
                p10 = np.clip(p10, 0.0, 1.0)
                p90 = np.clip(p90, 0.0, 1.0)

            metrics = metric_record(y_validation, raw_predictions, predictions)
            metric_row = {
                "model_run": "fixed_development_split",
                "target": target,
                "algorithm": algorithm,
                "train_wells": ",".join(training_wells),
                "validation_well": validation_well,
                "target_contract_status": config["contract_status"],
                "feature_panel": FEATURE_PANEL,
                "feature_count": len(feature_columns),
                "train_rows": len(X_train),
                "validation_scoreable_rows": len(X_validation),
                "sample_weight_supported": supports_weight,
                **metrics,
            }
            target_metric_rows.append(metric_row)
            candidate_models[algorithm] = model

            prediction = pd.DataFrame({
                "model_run": "fixed_development_split",
                "target": target,
                "algorithm": algorithm,
                "train_wells": ",".join(training_wells),
                "validation_well": validation_well,
                "well_alias": validation_well,
                "evaluation_role": "development_validation_well",
                "source_row": validation_frame.loc[scoreable, "source_row"].to_numpy(),
                "depth_original": validation_frame.loc[scoreable, "depth_original"].to_numpy(),
                "depth_original_unit": validation_frame.loc[scoreable, "depth_original_unit"].to_numpy(),
                "depth_m": validation_frame.loc[scoreable, "depth_m"].to_numpy(),
                "feature_count_present": present_count.loc[scoreable].to_numpy(),
                "feature_count_required": required_count,
                "y_true": y_validation.to_numpy(),
                "y_pred_raw": raw_predictions,
                "y_pred": predictions,
                "prediction_was_clipped": clipped_flags,
                "ensemble_p10": p10,
                "ensemble_p90": p90,
                "ensemble_spread_is_calibrated_interval": False,
            })
            target_prediction_frames.append(prediction)
            inventory_rows.append({
                "model_run": "fixed_development_split",
                "target": target,
                "algorithm": algorithm,
                "status": "evaluated_in_memory",
                "persisted": False,
                "train_wells": ",".join(training_wells),
                "validation_well": validation_well,
                "feature_count": len(feature_columns),
                "train_rows": len(X_train),
            })

        target_metrics = pd.DataFrame(target_metric_rows)
        mean_baseline_rmse = target_metrics.loc[
            target_metrics["algorithm"] == "mean_baseline", "rmse"
        ].iloc[0]
        target_metrics["rmse_skill_vs_mean_baseline"] = (
            1.0 - target_metrics["rmse"] / mean_baseline_rmse
            if pd.notna(mean_baseline_rmse) and mean_baseline_rmse > 0
            else np.nan
        )
        target_metrics = target_metrics.sort_values(
            ["rmse", "mae", "algorithm"], na_position="last"
        ).reset_index(drop=True)
        target_metrics["rank"] = np.arange(1, len(target_metrics) + 1)
        finite_ranking = target_metrics.dropna(subset=["rmse"])
        if finite_ranking.empty:
            raise RuntimeError(f"No model produced valid held-out metrics for {target}.")
        selected_algorithm = str(finite_ranking.iloc[0]["algorithm"])
        target_metrics["selected_model"] = target_metrics["algorithm"].eq(selected_algorithm)
        target_metrics["selection_basis"] = "lowest_RMSE_on_fixed_development_validation_well"
        metric_rows.extend(target_metrics.to_dict("records"))
        ranking_frames.append(target_metrics.copy())

        for prediction in target_prediction_frames:
            prediction["selected_model"] = prediction["algorithm"].eq(selected_algorithm)
            prediction_frames.append(prediction)

        selected_model = candidate_models[selected_algorithm]
        importance = heldout_permutation_importance(
            selected_model,
            X_validation,
            y_validation,
            feature_columns,
            target,
            validation_well,
            selected_algorithm,
        )
        if not importance.empty:
            importance_frames.append(importance)

        shift_rows.extend(feature_shift_records(
            train_data,
            validation_frame.loc[scoreable],
            feature_columns,
            target,
            "fixed_development_split",
            validation_well,
        ))
        selected_specs[target] = {
            "algorithm": selected_algorithm,
            "development_feature_columns": feature_columns,
            "development_train_wells": training_wells,
            "validation_well": validation_well,
            "target_contract_status": config["contract_status"],
        }

    return {
        "predictions": pd.concat(prediction_frames, ignore_index=True, sort=False) if prediction_frames else pd.DataFrame(),
        "metrics": pd.DataFrame(metric_rows),
        "model_ranking": pd.concat(ranking_frames, ignore_index=True, sort=False) if ranking_frames else pd.DataFrame(),
        "permutation_importance": pd.concat(importance_frames, ignore_index=True, sort=False) if importance_frames else pd.DataFrame(),
        "feature_policy": pd.concat(feature_policy_frames, ignore_index=True, sort=False) if feature_policy_frames else pd.DataFrame(),
        "feature_shift": pd.DataFrame(shift_rows),
        "model_inventory": pd.DataFrame(inventory_rows),
        "selected_specs": selected_specs,
    }


def run_optional_final_cross_well_audit(
    well_frames: dict[str, pd.DataFrame],
    selected_specs: dict[str, dict[str, Any]],
) -> dict[str, pd.DataFrame]:
    if not RUN_FINAL_CROSS_WELL_AUDIT:
        return {"predictions": pd.DataFrame(), "metrics": pd.DataFrame(), "feature_policy": pd.DataFrame()}

    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, Any]] = []
    policy_frames: list[pd.DataFrame] = []

    for target, selected in selected_specs.items():
        algorithm = selected["algorithm"]
        labeled_wells = [well for well, frame in well_frames.items() if _target_is_trainable(frame, target)]
        if len(labeled_wells) < 3:
            warnings.warn(f"Skipping final cross-well audit for {target}: fewer than three labeled wells.")
            continue
        for heldout_well in labeled_wells:
            training_wells = [well for well in labeled_wells if well != heldout_well]
            feature_columns, policy = select_training_features(well_frames, training_wells, heldout_well)
            policy.insert(0, "heldout_well", heldout_well)
            policy.insert(0, "train_wells", ",".join(training_wells))
            policy.insert(0, "target", target)
            policy.insert(0, "model_run", "final_cross_well_audit")
            policy_frames.append(policy)

            train_data = build_training_data(well_frames, training_wells, target, feature_columns)
            X_train = train_data[feature_columns]
            y_train = train_data[target]
            weights = equal_well_sample_weights(train_data)

            test_frame = well_frames[heldout_well]
            scoreable, present_count, required_count = row_score_mask(test_frame, feature_columns)
            X_test = test_frame.loc[scoreable, feature_columns]
            y_test = test_frame.loc[scoreable, target]
            if X_test.empty:
                continue

            model = make_model(algorithm)
            fit_model(model, X_train, y_train, weights)
            raw = model.predict(X_test)
            pred, clipped = bounded_predictions(raw)
            p10, p90 = ensemble_spread(model, X_test)
            if CLIP_SATURATION_PREDICTIONS:
                p10 = np.clip(p10, 0.0, 1.0)
                p90 = np.clip(p90, 0.0, 1.0)

            baseline = make_model("mean_baseline")
            fit_model(baseline, X_train, y_train, weights)
            baseline_raw = baseline.predict(X_test)
            baseline_pred, _ = bounded_predictions(baseline_raw)
            baseline_metrics = metric_record(y_test, baseline_raw, baseline_pred)
            metrics = metric_record(y_test, raw, pred)
            metrics["baseline_rmse"] = baseline_metrics["rmse"]
            metrics["rmse_skill_vs_mean_baseline"] = (
                1.0 - metrics["rmse"] / baseline_metrics["rmse"]
                if pd.notna(baseline_metrics["rmse"]) and baseline_metrics["rmse"] > 0 else np.nan
            )
            metric_rows.append({
                "model_run": "final_cross_well_audit",
                "target": target,
                "algorithm": algorithm,
                "train_wells": ",".join(training_wells),
                "heldout_well": heldout_well,
                "feature_panel": FEATURE_PANEL,
                "feature_count": len(feature_columns),
                "train_rows": len(X_train),
                **metrics,
            })
            prediction_frames.append(pd.DataFrame({
                "model_run": "final_cross_well_audit",
                "target": target,
                "algorithm": algorithm,
                "train_wells": ",".join(training_wells),
                "validation_well": heldout_well,
                "well_alias": heldout_well,
                "evaluation_role": "final_cross_well_audit_holdout",
                "source_row": test_frame.loc[scoreable, "source_row"].to_numpy(),
                "depth_original": test_frame.loc[scoreable, "depth_original"].to_numpy(),
                "depth_original_unit": test_frame.loc[scoreable, "depth_original_unit"].to_numpy(),
                "depth_m": test_frame.loc[scoreable, "depth_m"].to_numpy(),
                "feature_count_present": present_count.loc[scoreable].to_numpy(),
                "feature_count_required": required_count,
                "y_true": y_test.to_numpy(),
                "y_pred_raw": raw,
                "y_pred": pred,
                "prediction_was_clipped": clipped,
                "ensemble_p10": p10,
                "ensemble_p90": p90,
                "ensemble_spread_is_calibrated_interval": False,
                "selected_model": True,
            }))

    return {
        "predictions": pd.concat(prediction_frames, ignore_index=True, sort=False) if prediction_frames else pd.DataFrame(),
        "metrics": pd.DataFrame(metric_rows),
        "feature_policy": pd.concat(policy_frames, ignore_index=True, sort=False) if policy_frames else pd.DataFrame(),
    }


def fit_and_save_final_models(
    well_frames: dict[str, pd.DataFrame],
    selected_specs: dict[str, dict[str, Any]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    inventory_rows: list[dict[str, Any]] = []
    policy_frames: list[pd.DataFrame] = []
    if not FIT_FINAL_MODELS:
        return pd.DataFrame(), pd.DataFrame()

    for target, selected in selected_specs.items():
        algorithm = selected["algorithm"]
        all_labeled_wells = [well for well, frame in well_frames.items() if _target_is_trainable(frame, target)]
        feature_columns, policy = select_training_features(well_frames, all_labeled_wells, validation_well=None)
        policy.insert(0, "train_wells", ",".join(all_labeled_wells))
        policy.insert(0, "target", target)
        policy.insert(0, "model_run", "final_refit_all_labeled_wells")
        policy_frames.append(policy)

        train_data = build_training_data(well_frames, all_labeled_wells, target, feature_columns)
        X_train = train_data[feature_columns]
        y_train = train_data[target]
        weights = equal_well_sample_weights(train_data)
        model = make_model(algorithm)
        supports_weight = fit_model(model, X_train, y_train, weights)

        model_filename = f"{target}__{algorithm}__final.joblib"
        model_path = MODEL_RUN_ROOT / model_filename
        joblib.dump({
            "model": model,
            "run_id": RUN_ID,
            "target": target,
            "algorithm": algorithm,
            "feature_panel": FEATURE_PANEL,
            "feature_columns": feature_columns,
            "train_wells": all_labeled_wells,
            "target_contract_status": TARGET_CONFIG[target]["contract_status"],
            "row_completeness": {
                "minimum_features": MIN_FEATURES_PER_ROW,
                "minimum_fraction": MIN_FEATURE_FRACTION_PER_ROW,
            },
            "well_alias_policy": "WellA/WellB/WellC/WellD",
        }, model_path)
        inventory_rows.append({
            "model_run": "final_refit_all_labeled_wells",
            "target": target,
            "algorithm": algorithm,
            "status": "trained_and_persisted",
            "persisted": True,
            "model_file": str(model_path.relative_to(Path.cwd())) if model_path.is_relative_to(Path.cwd()) else model_filename,
            "train_wells": ",".join(all_labeled_wells),
            "feature_count": len(feature_columns),
            "train_rows": len(X_train),
            "sample_weight_supported": supports_weight,
        })

    return (
        pd.DataFrame(inventory_rows),
        pd.concat(policy_frames, ignore_index=True, sort=False) if policy_frames else pd.DataFrame(),
    )


## 5. Consolidated figures, workbook, prediction table, and review bundle

The export stage writes a small fixed set of files. The Excel workbook contains the audits and results as sheets; the PDF contains the selected-model figures as pages.

In [ ]:
# =============================================================================
# Consolidated figures and outputs
# =============================================================================

def _selected_prediction_rows(predictions: pd.DataFrame, target: str) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame()
    mask = (
        predictions["target"].eq(target)
        & predictions["model_run"].eq("fixed_development_split")
        & predictions["selected_model"].fillna(False)
    )
    return predictions.loc[mask].copy()


def generate_publication_pdf(
    predictions: pd.DataFrame,
    ranking: pd.DataFrame,
    importance: pd.DataFrame,
) -> pd.DataFrame:
    manifest: list[dict[str, Any]] = []
    page_number = 0
    png_root = RUN_ROOT / "paper_figures"
    if WRITE_PNG_FIGURES:
        png_root.mkdir(parents=True, exist_ok=True)

    plt.rcParams.update({
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
    })

    with PdfPages(FIGURES_PDF) as pdf:
        for target in ENABLED_TARGETS:
            selected = _selected_prediction_rows(predictions, target)
            scored = selected.dropna(subset=["y_true", "y_pred"])
            if not scored.empty:
                page_number += 1
                fig, ax = plt.subplots(figsize=(7.2, 7.2))
                ax.scatter(scored["y_true"], scored["y_pred"], s=12, alpha=0.55)
                lower = float(min(scored["y_true"].min(), scored["y_pred"].min()))
                upper = float(max(scored["y_true"].max(), scored["y_pred"].max()))
                ax.plot([lower, upper], [lower, upper], linewidth=1, label="1:1")
                metric = ranking.loc[(ranking["target"] == target) & ranking["selected_model"]].iloc[0]
                ax.text(
                    0.03,
                    0.97,
                    f"{metric['algorithm']}\n"
                    f"n = {int(metric['scored_rows'])}\n"
                    f"RMSE = {metric['rmse']:.3f}\n"
                    f"R² = {metric['r2']:.3f}",
                    transform=ax.transAxes,
                    va="top",
                    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
                )
                ax.set_title(f"Fixed held-out validation: {target}")
                ax.set_xlabel("Reference")
                ax.set_ylabel("Prediction")
                ax.legend(loc="lower right")
                fig.tight_layout()
                pdf.savefig(fig, bbox_inches="tight")
                filename = f"01_predicted_vs_reference_{target}.png"
                if WRITE_PNG_FIGURES:
                    fig.savefig(png_root / filename, dpi=300, bbox_inches="tight")
                plt.close(fig)
                manifest.append({
                    "pdf_page": page_number,
                    "figure_type": "fixed_holdout_predicted_vs_reference",
                    "target": target,
                    "png_file": filename if WRITE_PNG_FIGURES else "",
                })

            profile = selected.dropna(subset=["depth_m", "y_pred"]).sort_values("depth_m")
            if not profile.empty:
                page_number += 1
                fig, ax = plt.subplots(figsize=(7.2, 8.4))
                if profile["y_true"].notna().any():
                    ax.plot(profile["y_true"], profile["depth_m"], label="Reference", linewidth=1.2)
                ax.plot(profile["y_pred"], profile["depth_m"], label="Prediction", linewidth=1.2)
                if profile[["ensemble_p10", "ensemble_p90"]].notna().all(axis=1).any():
                    valid_spread = profile[["ensemble_p10", "ensemble_p90", "depth_m"]].dropna()
                    ax.fill_betweenx(
                        valid_spread["depth_m"],
                        valid_spread["ensemble_p10"],
                        valid_spread["ensemble_p90"],
                        alpha=0.2,
                        label="Tree ensemble P10–P90 spread",
                    )
                ax.invert_yaxis()
                ax.set_xlim(0, 1)
                ax.set_title(f"Validation depth profile: {target}")
                ax.set_xlabel("Saturation (v/v)")
                ax.set_ylabel("Depth (m)")
                ax.legend()
                fig.tight_layout()
                pdf.savefig(fig, bbox_inches="tight")
                filename = f"02_depth_profile_{target}.png"
                if WRITE_PNG_FIGURES:
                    fig.savefig(png_root / filename, dpi=300, bbox_inches="tight")
                plt.close(fig)
                manifest.append({
                    "pdf_page": page_number,
                    "figure_type": "fixed_holdout_depth_profile",
                    "target": target,
                    "png_file": filename if WRITE_PNG_FIGURES else "",
                })

            target_ranking = ranking.loc[ranking["target"] == target].sort_values("rmse")
            if not target_ranking.empty:
                page_number += 1
                fig, ax = plt.subplots(figsize=(8.2, 5.4))
                ax.barh(target_ranking["algorithm"], target_ranking["rmse"])
                ax.invert_yaxis()
                ax.set_title(f"Model comparison on the same validation well: {target}")
                ax.set_xlabel("RMSE (lower is better)")
                ax.set_ylabel("Algorithm")
                for y_position, (_, row) in enumerate(target_ranking.iterrows()):
                    marker = "  selected" if bool(row["selected_model"]) else ""
                    ax.text(row["rmse"], y_position, f"  R²={row['r2']:.2f}{marker}", va="center")
                fig.tight_layout()
                pdf.savefig(fig, bbox_inches="tight")
                filename = f"03_model_comparison_{target}.png"
                if WRITE_PNG_FIGURES:
                    fig.savefig(png_root / filename, dpi=300, bbox_inches="tight")
                plt.close(fig)
                manifest.append({
                    "pdf_page": page_number,
                    "figure_type": "fixed_holdout_model_comparison",
                    "target": target,
                    "png_file": filename if WRITE_PNG_FIGURES else "",
                })

            target_importance = importance.loc[importance["target"] == target].copy() if not importance.empty else pd.DataFrame()
            if not target_importance.empty:
                target_importance = target_importance.sort_values("importance_mean_rmse_increase").tail(15)
                page_number += 1
                fig, ax = plt.subplots(figsize=(8.2, 5.8))
                ax.barh(
                    target_importance["feature"],
                    target_importance["importance_mean_rmse_increase"],
                    xerr=target_importance["importance_std"],
                )
                ax.axvline(0, linewidth=0.8)
                ax.set_title(f"Held-out permutation importance: {target}")
                ax.set_xlabel("Increase in RMSE after permutation")
                ax.set_ylabel("Feature")
                fig.tight_layout()
                pdf.savefig(fig, bbox_inches="tight")
                filename = f"04_permutation_importance_{target}.png"
                if WRITE_PNG_FIGURES:
                    fig.savefig(png_root / filename, dpi=300, bbox_inches="tight")
                plt.close(fig)
                manifest.append({
                    "pdf_page": page_number,
                    "figure_type": "heldout_permutation_importance",
                    "target": target,
                    "png_file": filename if WRITE_PNG_FIGURES else "",
                })

        if page_number == 0:
            fig, ax = plt.subplots(figsize=(8.2, 4.5))
            ax.axis("off")
            ax.text(0.5, 0.5, "No scoreable model figures were produced.", ha="center", va="center")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            page_number = 1
            manifest.append({"pdf_page": 1, "figure_type": "no_figures_available", "target": "", "png_file": ""})

    return pd.DataFrame(manifest)


def write_prediction_table(predictions: pd.DataFrame) -> Path:
    try:
        predictions.to_parquet(PREDICTIONS_PARQUET, index=False)
        return PREDICTIONS_PARQUET
    except (ImportError, ModuleNotFoundError) as exc:
        fallback = RUN_ROOT / "all_predictions.csv.gz"
        warnings.warn(
            f"Parquet engine unavailable ({exc}). Wrote one compressed CSV instead: {fallback.name}"
        )
        predictions.to_csv(fallback, index=False, compression="gzip")
        return fallback


def _excel_safe_frame(frame: pd.DataFrame, note: str) -> pd.DataFrame:
    return frame if not frame.empty else pd.DataFrame({"note": [note]})


def write_results_workbook(tables: dict[str, pd.DataFrame]) -> None:
    with pd.ExcelWriter(RESULTS_WORKBOOK, engine="openpyxl") as writer:
        for sheet_name, frame in tables.items():
            safe_name = sheet_name[:31]
            _excel_safe_frame(frame, f"No rows produced for {sheet_name}.").to_excel(
                writer, sheet_name=safe_name, index=False
            )
        for worksheet in writer.book.worksheets:
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
            for column_cells in worksheet.columns:
                values = [str(cell.value) if cell.value is not None else "" for cell in column_cells[:200]]
                width = min(max(max((len(value) for value in values), default=0) + 2, 10), 45)
                worksheet.column_dimensions[column_cells[0].column_letter].width = width


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str | None:
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_versions() -> dict[str, str]:
    packages = ["numpy", "pandas", "scikit-learn", "matplotlib", "joblib", "openpyxl", "pyarrow"]
    versions: dict[str, str] = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not_installed"
    return versions


def build_run_summary(
    benchmark: dict[str, Any],
    final_audit: dict[str, pd.DataFrame],
    prediction_path: Path,
) -> pd.DataFrame:
    selected_rows = benchmark["model_ranking"].loc[
        benchmark["model_ranking"]["selected_model"]
    ] if not benchmark["model_ranking"].empty else pd.DataFrame()
    selected_text = "; ".join(
        f"{row.target}: {row.algorithm}"
        for row in selected_rows.itertuples()
    )
    rows = [
        ("run_id", RUN_ID),
        ("generated_at_utc", datetime.now(timezone.utc).isoformat(timespec="seconds")),
        ("workflow", "fixed_development_split"),
        ("enabled_targets", ",".join(ENABLED_TARGETS)),
        ("feature_panel", FEATURE_PANEL),
        ("candidate_models", ",".join(CANDIDATE_MODELS)),
        ("selected_models", selected_text),
        ("equal_well_sample_weights", EQUALIZE_WELL_WEIGHTS),
        ("minimum_feature_coverage", MIN_FEATURE_COVERAGE),
        ("minimum_features_per_row", MIN_FEATURES_PER_ROW),
        ("minimum_feature_fraction_per_row", MIN_FEATURE_FRACTION_PER_ROW),
        ("depth_used_as_feature", USE_DEPTH_AS_FEATURE),
        ("nmr_porosity_allowed", ALLOW_NMR_POROSITY_AS_FEATURE),
        ("provisional_A090_AF90_as_resistivity", ALLOW_PROVISIONAL_A090_AS_RESISTIVITY),
        ("predictions_clipped_to_0_1", CLIP_SATURATION_PREDICTIONS),
        ("final_cross_well_audit_ran", RUN_FINAL_CROSS_WELL_AUDIT),
        ("final_cross_well_audit_rows", len(final_audit.get("metrics", pd.DataFrame()))),
        ("prediction_file", prediction_path.name),
        ("results_workbook", RESULTS_WORKBOOK.name),
        ("figures_pdf", FIGURES_PDF.name),
        ("claims_boundary", "Fixed WellD results are development-validation evidence; run the optional final cross-well audit once after the design is frozen."),
    ]
    for target, config in TARGET_CONFIG.items():
        rows.extend([
            (f"{target}.enabled", config["enabled"]),
            (f"{target}.train_wells", ",".join(config["train_wells"])),
            (f"{target}.validation_well", config["validation_well"]),
            (f"{target}.contract_status", config["contract_status"]),
        ])
    return pd.DataFrame(rows, columns=["setting", "value"])


def write_run_manifest(prediction_path: Path, model_inventory: pd.DataFrame) -> dict[str, Any]:
    source_manifest = []
    for spec in SOURCE_SPECS:
        path = DATA_DIR / spec["filename"]
        source_manifest.append({
            "well_alias": spec["well_alias"],
            "filename": spec["filename"],
            "sheet": spec["sheet_name"],
            "exists": path.exists(),
            "sha256": sha256_file(path),
        })
    manifest = {
        "run_id": RUN_ID,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "package_versions": package_versions(),
        "data_directory_name": DATA_DIR.name,
        "source_files": source_manifest,
        "configuration": {
            "target_config": TARGET_CONFIG,
            "feature_panel": FEATURE_PANEL,
            "candidate_models": CANDIDATE_MODELS,
            "run_final_cross_well_audit": RUN_FINAL_CROSS_WELL_AUDIT,
            "equalize_well_weights": EQUALIZE_WELL_WEIGHTS,
            "clip_saturation_predictions": CLIP_SATURATION_PREDICTIONS,
            "random_state": RANDOM_STATE,
        },
        "outputs": {
            "model_results": RESULTS_WORKBOOK.name,
            "predictions": prediction_path.name,
            "paper_figures": FIGURES_PDF.name,
            "review_bundle": REVIEW_BUNDLE_ZIP.name if CREATE_REVIEW_BUNDLE else None,
        },
        "persisted_models": (
            model_inventory.loc[model_inventory.get("persisted", False) == True, "model_file"].dropna().tolist()
            if not model_inventory.empty and "model_file" in model_inventory else []
        ),
        "claims_boundary": (
            "The routine split is development validation. Candidate ranking is based on one fixed held-out well. "
            "Tree ensemble spread is not a calibrated predictive interval."
        ),
    }
    MANIFEST_JSON.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    latest = {
        "run_id": RUN_ID,
        "run_directory": str(RUN_ROOT.relative_to(Path.cwd())) if RUN_ROOT.is_relative_to(Path.cwd()) else str(RUN_ROOT),
        "manifest": str(MANIFEST_JSON.relative_to(Path.cwd())) if MANIFEST_JSON.is_relative_to(Path.cwd()) else str(MANIFEST_JSON),
    }
    LATEST_RUN_JSON.parent.mkdir(parents=True, exist_ok=True)
    LATEST_RUN_JSON.write_text(json.dumps(latest, indent=2), encoding="utf-8")
    return manifest


def build_review_bundle() -> Path | None:
    if not CREATE_REVIEW_BUNDLE:
        return None
    with zipfile.ZipFile(REVIEW_BUNDLE_ZIP, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in (RESULTS_WORKBOOK, FIGURES_PDF, MANIFEST_JSON):
            archive.write(path, arcname=path.name)
        archive.writestr(
            "README.txt",
            "North Slope validation review bundle\n\n"
            "Contains consolidated summary audits, metrics, model ranking, and publication figures.\n"
            "Does not contain source workbook rows, row-level predictions, or fitted model files.\n"
            "The fixed validation well is a development-validation set, not an untouched final test set.\n",
        )
    return REVIEW_BUNDLE_ZIP


def run_modeling_and_exports(
    well_frames: dict[str, pd.DataFrame],
    audits: dict[str, pd.DataFrame],
    target_audit: pd.DataFrame,
    readiness: pd.DataFrame,
    feature_coverage: pd.DataFrame,
    leakage: pd.DataFrame,
) -> dict[str, Any]:
    prepare_run_directories()
    benchmark = run_fixed_development_benchmark(well_frames)
    final_audit = run_optional_final_cross_well_audit(well_frames, benchmark["selected_specs"])
    final_model_inventory, final_feature_policy = fit_and_save_final_models(
        well_frames, benchmark["selected_specs"]
    )

    prediction_parts = [benchmark["predictions"]]
    if not final_audit["predictions"].empty:
        prediction_parts.append(final_audit["predictions"])
    predictions = pd.concat(prediction_parts, ignore_index=True, sort=False)
    prediction_path = write_prediction_table(predictions)

    figure_manifest = generate_publication_pdf(
        predictions,
        benchmark["model_ranking"],
        benchmark["permutation_importance"],
    )

    model_inventory = pd.concat(
        [benchmark["model_inventory"], final_model_inventory],
        ignore_index=True,
        sort=False,
    )
    feature_policy = pd.concat(
        [benchmark["feature_policy"], final_audit["feature_policy"], final_feature_policy],
        ignore_index=True,
        sort=False,
    )
    run_summary = build_run_summary(benchmark, final_audit, prediction_path)

    workbook_tables = {
        "run_summary": run_summary,
        "heldout_metrics": benchmark["metrics"],
        "model_ranking": benchmark["model_ranking"],
        "final_audit_metrics": final_audit["metrics"],
        "target_audit": target_audit,
        "well_readiness": readiness,
        "feature_coverage": feature_coverage,
        "feature_policy": feature_policy,
        "feature_shift": benchmark["feature_shift"],
        "leakage_audit": leakage,
        "perm_importance": benchmark["permutation_importance"],
        "model_inventory": model_inventory,
        "source_layout": audits["source_layout"],
        "column_mapping": audits["column_mapping"],
        "unit_conversion": audits["unit_conversion"],
        "refined_sheets": audits["refined_sheet_inventory"],
        "ignored_inputs": audits["ignored_inputs"],
        "figure_manifest": figure_manifest,
    }
    write_results_workbook(workbook_tables)

    if WRITE_STANDARDIZED_TABLE:
        standardized_path = RUN_ROOT / "standardized_wells.parquet"
        audits["combined"].to_parquet(standardized_path, index=False)
    else:
        standardized_path = None

    manifest = write_run_manifest(prediction_path, model_inventory)
    review_bundle = build_review_bundle()
    return {
        "benchmark": benchmark,
        "final_audit": final_audit,
        "predictions": predictions,
        "prediction_path": prediction_path,
        "figure_manifest": figure_manifest,
        "model_inventory": model_inventory,
        "feature_policy": feature_policy,
        "run_summary": run_summary,
        "manifest": manifest,
        "review_bundle": review_bundle,
        "standardized_path": standardized_path,
    }


def run_full_pipeline(data_dir: Path | None = None) -> dict[str, Any]:
    global DATA_DIR
    if data_dir is not None:
        DATA_DIR = Path(data_dir).expanduser()
    well_frames, audits = standardize_all_sources(DATA_DIR)
    target_audit, readiness, feature_coverage, leakage = build_target_and_readiness_audits(well_frames)
    modeling = run_modeling_and_exports(
        well_frames,
        audits,
        target_audit,
        readiness,
        feature_coverage,
        leakage,
    )
    return {
        "well_frames": well_frames,
        "audits": audits,
        "target_audit": target_audit,
        "readiness": readiness,
        "feature_coverage": feature_coverage,
        "leakage": leakage,
        **modeling,
    }


## 6. Preflight: rebuild and audit four standardized wells

This cell does not create per-well CSVs. It prepares the in-memory tables and displays the checks that must pass before modeling.

In [ ]:
print("Data folder:", DATA_DIR)
print("Run ID:", RUN_ID)
print("Run folder:", RUN_ROOT)
print("\nConfigured routine split:")
for target, config in TARGET_CONFIG.items():
    print(
        f"  {target}: enabled={config['enabled']} | "
        f"train={list(config['train_wells'])} -> validate={config['validation_well']}"
    )

print("\nRequired original workbooks:")
for spec in SOURCE_SPECS:
    path = DATA_DIR / spec["filename"]
    print(f"  {spec['well_alias']}: {spec['filename']} ->", "FOUND" if path.exists() else "MISSING")

well_frames, audits = standardize_all_sources(DATA_DIR)
target_audit, readiness, feature_coverage, leakage = build_target_and_readiness_audits(well_frames)

print("\nSTANDARDIZED WELL SHAPES")
for well_alias, frame in well_frames.items():
    print(
        well_alias,
        frame.shape,
        "original depth:",
        round(float(frame["depth_original"].min()), 3),
        "to",
        round(float(frame["depth_original"].max()), 3),
        str(frame["depth_original_unit"].dropna().iloc[0]),
    )

print("\nSOURCE LAYOUT")
display(audits["source_layout"])
print("\nTARGET AUDIT")
display(target_audit)
print("\nWELL READINESS")
display(readiness)
print("\nACTIVE FEATURE-PANEL COVERAGE")
display(feature_coverage.pivot(index="feature", columns="well_alias", values="coverage"))


## 7. Run the fixed benchmark, select one model, and export consolidated results

This routine cell tests all candidate algorithms on the same configured validation well. It does not create leave-one-well-out fold models. Set `RUN_FINAL_CROSS_WELL_AUDIT = True` only for the one-time final audit after the workflow is frozen.

In [ ]:
results = run_modeling_and_exports(
    well_frames,
    audits,
    target_audit,
    readiness,
    feature_coverage,
    leakage,
)

print("\nMODEL RANKING — SAME FIXED VALIDATION WELL")
display(results["benchmark"]["model_ranking"])

print("\nSELECTED FINAL MODEL INVENTORY")
display(results["model_inventory"].loc[results["model_inventory"].get("persisted", False) == True])

if not results["final_audit"]["metrics"].empty:
    print("\nONE-TIME FINAL CROSS-WELL AUDIT")
    display(results["final_audit"]["metrics"])

print("\nCONSOLIDATED OUTPUTS")
print("  Results workbook:", RESULTS_WORKBOOK)
print("  Predictions:", results["prediction_path"])
print("  Figures PDF:", FIGURES_PDF)
print("  Manifest:", MANIFEST_JSON)
print("  Review bundle:", results["review_bundle"])
print("  Final model folder:", MODEL_RUN_ROOT)


## Output interpretation

Use `model_results.xlsx` as the main review file:

- `model_ranking` compares all algorithms on the same fixed validation well.
- `heldout_metrics` includes bounded and raw metrics, clipping rate, and skill versus the equal-well mean baseline.
- `feature_policy` proves that feature selection used training wells only.
- `perm_importance` is calculated on the fixed held-out well for the selected non-baseline model.
- `feature_shift` identifies validation values outside the training range.
- `target_audit` and `leakage_audit` preserve the scientific caveats.

`all_predictions.parquet` is local and row-level. `north_slope_validation_bundle.zip` intentionally omits it, along with source rows and model files.

The default `WellD` result is a **development-validation result**. Do not describe it as an untouched final test after repeatedly changing features or algorithms in response to its score. Once the design is frozen, run `RUN_FINAL_CROSS_WELL_AUDIT = True` a single time and report that audit separately.
